# 14.1 MuJoCo와 정책 학습 실습 — 노트북

[![Open In Colab: MuJoCo 진자 제어](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter14_1_mujoco_pd_pendulum.ipynb)

책 본문: [14.1 MuJoCo와 정책 학습 실습](https://smhanlab.com/book-ml/kor/ml2/chapter14.html)

이 노트북은 책 14.1절의 MuJoCo 코드를 그대로 실행합니다:
(1) 자유낙하로 `qpos`/`mj_step`의 기본 API를 확인하고,
(2) **PD 제어**와 **무작위 제어**로 진자를 비교(그림),
(3) "자주 하는 실수" — D항(속도 피드백)을 빼거나 부호를 뒤집으면 어떻게 되는지,
(4) raw MuJoCo를 Gymnasium 스타일 `reset()/step()` 환경으로 감싸
Chapter 9~13의 알고리즘을 그대로 달 수 있음을 보여주고,
(5) **접촉(부딪혀 튀는)** 이 수치적으로 깨지지 않고 안정적으로 계산되는지 봅니다.

## 0. 설정: 한국어 폰트, 시드 고정, 이미지 저장 경로

In [1]:
import os
import numpy as np
import mujoco
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 한국어 라벨 (없으면 기본 폰트로 넘어가도 출력은 됨)
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr:
    plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print("mujoco", mujoco.__version__, " | 그림 저장 위치:", IMG)

mujoco 3.12.0  | 그림 저장 위치: /home/smhan/book-ml/kor/src/images


## 1. 자유낙하: `MjModel` / `MjData` / `mj_step`의 기본 API

공 하나를 자유 공간에 띄워 둔다. `free` 관절의 `qpos`가 **7**개(위치 3 + 방향 4 쿼터니언),
`qvel`이 **6**개(선속도 3 + 각속도 3)인 이유를 먼저 확인하고, 5·50스텝의 높이 변화를
재본다. 50스텝(≈0.1s)이면 자유낙하 공식 \(h=1-\tfrac12gt^2\)의 0.951m과 거의 같아야
한다.

In [2]:
xml_free = """
<mujoco>
  <worldbody>
    <light name="top" pos="0 0 1"/>
    <geom type="plane" size="1 1 0.1"/>
    <body pos="0 0 1">
      <joint type="free"/>
      <geom type="sphere" size="0.1" rgba="1 0 0 1"/>
    </body>
  </worldbody>
</mujoco>
"""
m = mujoco.MjModel.from_xml_string(xml_free)
d = mujoco.MjData(m)
print("위치 자유도(nq):", m.nq, " 속도 자유도(nv):", m.nv)

h = []
for s in range(51):
    if s > 0:
        mujoco.mj_step(m, d)
    h.append(d.qpos[2])
print("5스텝 후 높이 :", round(h[5], 4))
print("50스텝 후 높이:", round(h[50], 4), " (자유낙하 공식 예측 ≈ 0.951)")

위치 자유도(nq): 7  속도 자유도(nv): 6
5스텝 후 높이 : 0.9994
50스텝 후 높이: 0.95  (자유낙하 공식 예측 ≈ 0.951)


## 2. 진자: PD 제어 vs 무작위 제어

힌지(각도 \(\theta\)) 하나 + 그 관절에 토크를 가하는 `motor` 액추에이터.
초기 각도를 목표(0)에서 멀리 떨어진 \(\theta_0=3.0\)에서 시작해 200스텝 굴린다.
**PD**는 \(\text{토크}=-3\theta-0.5\dot\theta\), **무작위**는 매 스텝
\(U[-5,5]\)를 넣는다. (책 본문과 동일한 시드/매개변수 — seed 0.)

In [3]:
xml = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 2"/>
    <body pos="0 0 1">
      <joint name="hinge" type="hinge" axis="0 1 0" damping="0.1"/>
      <geom type="capsule" fromto="0 0 0  0 0 -0.5" size="0.02" mass="1"/>
    </body>
  </worldbody>
  <actuator>
    <motor joint="hinge" gear="1" ctrlrange="-5 5"/>
  </actuator>
</mujoco>
"""

def run_pendulum(policy, seed, steps=200):
    model = mujoco.MjModel.from_xml_string(xml)
    data = mujoco.MjData(model)
    data.qpos[0] = 3.0
    mujoco.mj_forward(model, data)
    rng = np.random.default_rng(seed)
    traj = []
    for _ in range(steps):
        th, thd = data.qpos[0], data.qvel[0]
        if policy == "pd":
            data.ctrl[0] = float(np.clip(-3.0*th - 0.5*thd, -5, 5))
        else:
            data.ctrl[0] = float(rng.uniform(-5, 5))
        mujoco.mj_step(model, data)
        traj.append(data.qpos[0])
    return traj

pd_traj     = run_pendulum("pd", 0)
random_traj = run_pendulum("random", 0)

print("무작위 정책: 평균 |각도| =", round(float(np.mean(np.abs(random_traj))), 2),
      " (초기 3.0에서 거의 안 줄어듦)")
print("PD 정책    : 평균 |각도| =", round(float(np.mean(np.abs(pd_traj))), 2),
      ", 최종 각도 =", round(pd_traj[-1], 4))

무작위 정책: 평균 |각도| = 2.97  (초기 3.0에서 거의 안 줄어듦)
PD 정책    : 평균 |각도| = 1.69 , 최종 각도 = -0.0645


두 궤적: 파란 실선(PD)은 진폭이 갈수록 작아져 목표 0에 안착하고,
회색 선(무작위)는 \(\pm\pi\) 안을 떠돌며 \(\theta_0=3.0\) 근처에 머문다.

In [4]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(random_traj, color="gray", linewidth=1.0, label="무작위 제어")
ax.plot(pd_traj, color="tab:blue", linewidth=1.4, label="PD 제어  (-3θ - 0.5θ̇)")
ax.axhline(0, color="tab:red", linestyle="--", linewidth=1, label="목표 (θ=0)")
ax.set_xlabel("스텝"); ax.set_ylabel("각도 θ (라디안)")
ax.set_title("MuJoCo 진자: PD 제어 vs 무작위 제어 (200스텝)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch14_1_pendulum_pd_vs_random.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch14_1_pendulum_pd_vs_random.svg")

저장: /home/smhan/book-ml/kor/src/images/ch14_1_pendulum_pd_vs_random.svg


## 3. "자주 하는 실수": D항을 빼거나 부호를 뒤집으면

PD가 마법인 게 아니라 **D항(\(-0.5\dot\theta\))**이 마법이라는 것을
이득만 바꿔가며 확인한다. 각 정책으로 200스텝 굴려 200스텝 후의
\(\|\theta\|\)와 \(\|\dot\theta\|\)를 재본다.

In [5]:
def run_gain(kp, kd, steps=200):
    model = mujoco.MjModel.from_xml_string(xml)
    data = mujoco.MjData(model)
    data.qpos[0] = 3.0
    mujoco.mj_forward(model, data)
    for _ in range(steps):
        th, thd = data.qpos[0], data.qvel[0]
        data.ctrl[0] = float(np.clip(kp*th + kd*thd, -5, 5))
        mujoco.mj_step(model, data)
    return abs(data.qpos[0]), abs(data.qvel[0])

cases = [
    ("완전한 PD   (-3θ - 0.5θ̇)", -3.0, -0.5),
    ("P만          (-3θ)",        -3.0,  0.0),
    ("너무 약한 PD (-1θ - 0.2θ̇)", -1.0, -0.2),
    ("부호 오류    (+3θ - 0.5θ̇)",  3.0, -0.5),
    ("제어 없음          (0)",      0.0,  0.0),
]
print(f"{'정책':28s} {'|θ|':>7s} {'|θ̇|':>7s}")
for name, kp, kd in cases:
    a_th, a_thd = run_gain(kp, kd)
    print(f"{name:28s} {a_th:7.3f} {a_thd:7.3f}")

정책                               |θ|    |θ̇|
완전한 PD   (-3θ - 0.5θ̇)         0.065   6.257
P만          (-3θ)              1.306  11.514
너무 약한 PD (-1θ - 0.2θ̇)         0.689   9.194
부호 오류    (+3θ - 0.5θ̇)         7.621  20.999
제어 없음          (0)             2.604   2.466


## 4. 물리 시뮬레이터를 Gymnasium "환경"으로 감싸기

`reset()`/`step()`만 구현하면 Chapter 9~13의 알고리즘이 *그대로*
달린다. 이 30줄짜리 래퍼가 `Reacher-v5`/`Ant-v5`가 내부적으로
하는 일의 *미니판*이다. 이 래퍼로 PD·무작위 정책을 굴리면,
2절에서 `mj_step`을 직접 돌린 것과 **완전히 같은** 결과가 나와야
한다 — 바로 이 "한 줄만 바꿔도 된다"를 확인한다.

In [6]:
class MuJoCoPendulum:
    """raw MuJoCo 진자를 Gymnasium 스타일의 (s, a, r, done) 환경으로 감싼다."""
    def __init__(self, seed=0):
        self.model = mujoco.MjModel.from_xml_string(xml)
        self.data = mujoco.MjData(self.model)
        self.rng = np.random.default_rng(seed)
        self.max_steps = 200

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.data.qpos[0] = 3.0
        self.data.qvel[0] = 0.0
        mujoco.mj_forward(self.model, self.data)
        self.t = 0
        return np.array([self.data.qpos[0], self.data.qvel[0]]), {}

    def step(self, action):
        self.data.ctrl[0] = float(np.clip(action, -5, 5))  # 토크 한도
        mujoco.mj_step(self.model, self.data)
        th = self.data.qpos[0]
        reward = -th * th                 # 0에 가까울수록 보상 큼
        self.t += 1
        done = self.t >= self.max_steps
        s = np.array([self.data.qpos[0], self.data.qvel[0]])
        return s, float(reward), bool(done), False, {}

def drive(env, policy, seed):
    s, _ = env.reset(seed=seed)
    rng = np.random.default_rng(seed)
    total = 0.0; abs_th = []
    for _ in range(200):
        a = float(np.clip(-3.0*s[0] - 0.5*s[1], -5, 5)) if policy == "pd" \
            else float(rng.uniform(-5, 5))
        s, r, done, _, _ = env.step(a)
        total += r; abs_th.append(abs(s[0]))
    return s, total, float(np.mean(abs_th))

env = MuJoCoPendulum()
s_pd, r_pd, m_pd   = drive(env, "pd", 0)
s_rnd, r_rnd, m_rnd = drive(env, "random", 0)
print("PD (env.step): 최종 (θ, θ̇) =", np.round(s_pd, 4),
      " 누적 보상 =", round(r_pd, 1))
print("무작위(env.step): 평균 |θ| =", round(m_rnd, 3))
print()
print("→ 2절의 직접 mj_step 결과와 동일: PD 최종 각도 -0.065, 무작위 평균|θ|≈2.9")
print("→ 즉, 알고리즘이 env.reset()/env.step()만 쓰면 물리엔진이 MuJoCo로")
print("  바뀌어도 정책 코드는 한 줄(env 인자)만 바꿔 그대로 재사용된다.")

PD (env.step): 최종 (θ, θ̇) = [-0.0645 -6.2568]  누적 보상 = -777.0
무작위(env.step): 평균 |θ| = 2.971

→ 2절의 직접 mj_step 결과와 동일: PD 최종 각도 -0.065, 무작위 평균|θ|≈2.9
→ 즉, 알고리즘이 env.reset()/env.step()만 쓰면 물리엔진이 MuJoCo로
  바뀌어도 정책 코드는 한 줄(env 인자)만 바꿔 그대로 재사용된다.


## 5. "정밀"의 뿌리: 접촉(부딪혀 튀는)이 수치적으로 깨지지 않는다

공을 1m 높이에서 떨어뜨려 바닥(`plane`)에 닿게 한다. "공이 바닥에 **부딪힌다**"는
순간은 단순 미분방정식이 아니다 — 두 물체가 겹쳐 들어가면 안 되면서(비침투),
겹치지 않을 때는 힘을 주지 않아도 된다(비음)는 조건이 *동시에* 만족되어야 하므로
**LCP(선형 보완 문제)**라는 특별한 최적화다. MuJoCo는 매 스텝 이 LCP를 **안정적으로**
풀어내 (1) 공이 부딪혀 튀다 멈추는 행위가 NaN·폭주 없이 자연스럽고, (2) 다리 로봇
발이 바닥에 붙었다 떨어지는 걸음 같은 복잡한 접촉도 같은 식으로 처리된다.

> **솔직한 비고: "비침투"가 완전히 성립하는 건 아니다.** 아래를 보면 충돌 직후
> 공의 **하단**이 바닥 아래로 약 0.03m 가라앉았다가 돌아온다. MuJoCo 접촉은
> *soft contact*(탄성 접촉)이라 완전한 "뚫고 안 들어감" 제약이 아니라 살짝
> 가라앉는(=변형) 것을 허용한다 — 실제 물체도 부딪히면 변형된다. "정밀"의 핵심은
> 침투가 전혀 없는 게 아니라, **이 접촉을 풀어내도 시뮬레이션이 수치적으로
> 깨지지 않고(안정수렴) 공이 바닥 위에(중심 높이 = 반지름 0.05m) 조용히 안착**하는
> 것이다.

In [7]:
xml_ball = """
<mujoco>
  <option timestep="0.002" gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 1 2"/>
    <geom type="plane" size="1 1 0.1" rgba="0.8 0.8 0.9 1"/>
    <body pos="0 0 1.0">
      <joint type="free"/>
      <geom type="sphere" size="0.05" rgba="1 0 0 1" mass="0.5"/>
    </body>
  </worldbody>
</mujoco>
"""
mb = mujoco.MjModel.from_xml_string(xml_ball)
db = mujoco.MjData(mb)
db.qpos[2] = 1.0
mujoco.mj_forward(mb, db)
heights = []
for s in range(300):
    mujoco.mj_step(mb, db)
    heights.append(db.qpos[2])

bottom = [x - 0.05 for x in heights]   # 공의 하단(바닥에서 높이)
print("150스텝 후 높이(중심):", round(heights[149], 3), " (바닥에 닿아 튀는 중)")
print("299스텝 후 높이(중심):", round(heights[298], 3), " (= 반지름 0.05 → 하단이 바닥에 딱 닿아 안착)")
print("충돌 직후 하단 최저점:", round(min(bottom), 3), " m  (soft contact라 약 0.03m 가라앉았다가 복원 — 변형)")
print("→ 시뮬레이션은 NaN·폭주 없이 안정적, 공은 바닥 위에 조용히 멈춘다")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, 301), bottom, color="tab:red", linewidth=1.4)
ax.axhline(0, color="black", linewidth=1, label="바닥 (plane)")
ax.set_xlabel("스텝"); ax.set_ylabel("공 하단의 높이 (m)")
ax.set_title("접촉 시뮬레이션: 1m 낙하 → 부딪혀 튀다 멈춤 (LCP, soft contact)")
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_ylim(-0.06, 1.05)
fig.tight_layout()
fig.savefig(IMG + "/ch14_1_contact_bounce.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch14_1_contact_bounce.svg")

150스텝 후 높이(중심): 0.556  (바닥에 닿아 튀는 중)
299스텝 후 높이(중심): 0.05  (= 반지름 0.05 → 하단이 바닥에 딱 닿아 안착)
충돌 직후 하단 최저점: -0.029  m  (soft contact라 약 0.03m 가라앉았다가 복원 — 변형)
→ 시뮬레이션은 NaN·폭주 없이 안정적, 공은 바닥 위에 조용히 멈춘다
저장: /home/smhan/book-ml/kor/src/images/ch14_1_contact_bounce.svg


### 정리

- **`qpos`/`qvel`/`ctrl`**이 MDP의 상태/보상과 어떻게 대응하는지 직접
  API로 확인했다.
- **PD의 D항**(속도 피드백)이 없으면 진동/발산 — 물리적으로 안정한
  정책이 먼저 있어야 강화학습이 배울 것이다.
- **30줄 래퍼**로 MuJoCo를 `reset()/step()` 환경으로 감싸면,
  Chapter 9~13의 알고리즘이 한 줄(env 인자)만 바꿔 그대로 달린다.
- **접촉**은 LCP(soft contact)로 풀려서 시뮬레이션이 수치적으로
  깨지지 않는다 — 14.2·14.3에서 "정밀 물리엔진이 왜 필요한가"의 뿌리.